In [ ]:
def get_posts(target_username, login_username=None, login_password=None, top_k=1):
    """
    Extract top_k Instagram posts as in-memory data.
    Returns: list of dicts with caption, metadata, images (bytes)
    """

    import instaloader
    import requests
    from itertools import islice

    L = instaloader.Instaloader()

    if login_username and login_password:
        L.login(login_username, login_password)

    profile = instaloader.Profile.from_username(
        L.context,
        target_username
    )

    posts_data = []

    session = requests.Session()
    session.headers.update({
        "User-Agent": "Mozilla/5.0"
    })

    for post in islice(profile.get_posts(), top_k):

        caption = post.caption
        metadata = post._node  # internal but useful

        image_urls = []

        if post.typename == "GraphImage":
            image_urls.append(post.url)

        elif post.typename == "GraphSidecar":
            for node in post.get_sidecar_nodes():
                image_urls.append(node.display_url)

        images = []
        for url in image_urls:
            resp = session.get(url, timeout=10)
            resp.raise_for_status()
            images.append(resp.content)

        posts_data.append({
            "caption": caption,
            "metadata": metadata,
            "images": images
        })

    return posts_data


In [ ]:
posts_data = get_posts("maxverstappen1")

In [ ]:
posts_data[0]["caption"]

In [3]:
import cv2
import os
from insightface.utils import face_align
from insightface.app import FaceAnalysis

providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']
face_analyzer = FaceAnalysis(name='antelopev2', providers=providers)
face_analyzer.prepare(ctx_id=0, det_size=(640, 640))

def detect_faces_from_image_path(image_path, face_analyzer):
    """
    Detect faces from an image path using InsightFace.

    Returns a list of dicts with:
    - face_img      : aligned face (BGR)
    - display_img  : raw cropped face (BGR)
    - bbox         : (x, y, w, h)
    - embedding    : ArcFace embedding (np.float32)
    - det_score    : detection confidence
    """

    if not os.path.exists(image_path):
        raise FileNotFoundError(f"Image not found: {image_path}")

    frame = cv2.imread(image_path)
    if frame is None:
        raise ValueError(f"Failed to load image: {image_path}")

    h, w = frame.shape[:2]
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    results = []

    try:
        faces = face_analyzer.get(rgb_frame)

        for face in faces:
            x1, y1, x2, y2 = face.bbox.astype(int)

            # Clip bbox
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(w, x2), min(h, y2)

            if x2 <= x1 or y2 <= y1:
                continue

            display_img = frame[y1:y2, x1:x2]

            # Alignment
            if face.kps is not None:
                aligned = face_align.norm_crop(rgb_frame, face.kps)
                face_img = cv2.cvtColor(aligned, cv2.COLOR_RGB2BGR)
            else:
                face_img = display_img

            results.append({
                "face_img": face_img,
                "display_img": display_img,
                "bbox": (x1, y1, x2 - x1, y2 - y1),
                "embedding": face.embedding.astype("float32"),
                "det_score": float(face.det_score)
            })

        return results

    except Exception as e:
        print(f"[Face Detection Error] {e}")
        return []


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\bryan/.insightface\models\antelopev2\1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\bryan/.insightface\models\antelopev2\2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\bryan/.insightface\models\antelopev2\genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\bryan/.insightface\models\antelopev2\glintr100.onnx recognition ['None', 3, 112, 112] 127.5 127.5
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\bryan/.insightface\models\antelopev2\scrfd_10g_bnkps.onnx detection [1, 3, '

In [ ]:
faces = detect_faces_from_image_path("1.jpg", face_analyzer)

for i, f in enumerate(faces):
    print(f"Face {i}: score={f['det_score']}, bbox={f['bbox']}")


In [2]:
import cv2
import os

def save_faces_with_bboxes(
    image_path,
    face_analyzer,
    output_path="output.jpg"
):
    """
    Detect faces in an image and save the image with bounding boxes drawn.

    Args:
        image_path (str): input image path
        face_analyzer: initialized InsightFace FaceAnalysis
        output_path (str): optional output path

    Returns:
        str: path to saved image
    """

    faces = detect_faces_from_image_path(image_path, face_analyzer)

    image = cv2.imread(image_path)
    if image is None:
        raise ValueError("Failed to load image")

    for face in faces:
        x, y, w, h = face["bbox"]
        score = face["det_score"]

        cv2.rectangle(
            image,
            (x, y),
            (x + w, y + h),
            (0, 255, 0),
            2
        )

        cv2.putText(
            image,
            f"{score:.2f}",
            (x, y - 6),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (0, 255, 0),
            1
        )

    if output_path is None:
        base, ext = os.path.splitext(image_path)
        output_path = f"{base}_faces{ext}"

    cv2.imwrite(output_path, image)
    print(f"Saved to {output_path}")


In [5]:
save_faces_with_bboxes("1.jpg", face_analyzer)

d:\Projects\Research Work\Safe Scroll\venv\Lib\site-packages\insightface\utils\face_align.py:23: FutureWarning: `estimate` is deprecated since version 0.26 and will be removed in version 2.2. Please use `SimilarityTransform.from_estimate` class constructor instead.
  tform.estimate(lmk, dst)


Saved to output.jpg
